# Exploring the Dataset: Intranet Server Auth Log

**Goal:** Understand the structure of `intranet_server/logs/auth.log` (Linux auth/syslog) to design the `auth_events` table.

This notebook walks through:
1. Loading the raw auth log (272 lines of syslog-style `Month Day Time hostname process[pid]: message`)
2. Parsing the format and exploring every field (including message sub-fields for analysis)
3. Building a raw 1:1 DataFrame (`df_raw`) for DDL; keeping parsed `message_*` columns in `df` for field-by-field exploration only
4. Integrating ground truth labels (8 labeled lines, privilege escalation)
5. Mapping to the planned raw table schema with both PostgreSQL and MySQL DDL
6. Checking for 1NF, 2NF, and 3NF violations using the `normalization_rules_sheet.md` checklist

---

**Dataset:** AIT Log Data Set V2.0 — russellmitchell testbed
**Source:** https://zenodo.org/records/5789064
**Host:** intranet_server (main attack target, WordPress/intranet)


## 0. Configuration

Set the path to the `russellmitchell/` dataset folder below.

**Default:** Assumes `russellmitchell/` is at the same level as the repo:
```
data-201-group-project/
  data-201-security-log-analysis/   <-- this repo
    notebooks/                       <-- this notebook is here
  russellmitchell/                   <-- dataset is here
```

If your dataset is somewhere else, change `DATASET_ROOT` below. We also need the label file path for ground-truth attack lines.


In [1]:
from pathlib import Path

# --- CHANGE THIS if your dataset is in a different location ---
DATASET_ROOT = Path("..") / ".." / "russellmitchell"
AUTH_LOG = DATASET_ROOT / "gather" / "intranet_server" / "logs" / "auth.log"
LABEL_FILE = DATASET_ROOT / "labels" / "intranet_server" / "logs" / "auth.log"

for name, path in [
    ("Dataset root", DATASET_ROOT),
    ("Auth log", AUTH_LOG),
    ("Label file", LABEL_FILE),
]:
    status = "FOUND" if path.exists() else "MISSING"
    print(f"{name}: {path.resolve()} [{status}]")

Dataset root: /Users/rrosevearehunt/Education/DATA201/GroupProject/russellmitchell [FOUND]
Auth log: /Users/rrosevearehunt/Education/DATA201/GroupProject/russellmitchell/gather/intranet_server/logs/auth.log [FOUND]
Label file: /Users/rrosevearehunt/Education/DATA201/GroupProject/russellmitchell/labels/intranet_server/logs/auth.log [FOUND]


## 1. Load Raw Data

The auth log uses syslog-style lines:
```
Month Day HH:MM:SS hostname process[pid]: message
```
Example: `Jan 23 06:25:05 intranet-server CRON[22883]: pam_unix(cron:session): session closed for user root`

Load all lines with 1-based line numbers (to match the label file).


In [2]:
# Load all lines with 1-based line numbers (matching label file convention)
with open(AUTH_LOG) as f:
    raw_lines = f.readlines()

print(f"Total lines: {len(raw_lines)}")
print()
print("First 5 lines:")
for i, line in enumerate(raw_lines[:5], 1):
    print(f"  [{i}] {line.rstrip()}")
print()
print("Last 3 lines:")
for i, line in enumerate(raw_lines[-3:], len(raw_lines) - 2):
    print(f"  [{i}] {line.rstrip()}")

Total lines: 272

First 5 lines:
  [1] Jan 23 06:25:05 intranet-server CRON[22883]: pam_unix(cron:session): session closed for user root
  [2] Jan 23 06:39:01 intranet-server CRON[23064]: pam_unix(cron:session): session opened for user root by (uid=0)
  [3] Jan 23 06:39:01 intranet-server CRON[23064]: pam_unix(cron:session): session closed for user root
  [4] Jan 23 06:47:01 intranet-server CRON[23137]: pam_unix(cron:session): session opened for user root by (uid=0)
  [5] Jan 23 06:47:02 intranet-server CRON[23137]: pam_unix(cron:session): session closed for user root

Last 3 lines:
  [270] Jan 24 23:17:01 intranet-server CRON[31443]: pam_unix(cron:session): session closed for user root
  [271] Jan 24 23:39:01 intranet-server CRON[31519]: pam_unix(cron:session): session opened for user root by (uid=0)
  [272] Jan 24 23:39:01 intranet-server CRON[31519]: pam_unix(cron:session): session closed for user root


## 2. Parse the Auth Log Format

### 2.1 Process name (event family) inventory

First, extract all distinct process names and their counts. These correspond to event families (CRON, sshd, sudo, su, systemd-logind, systemd).


In [8]:
import re
from collections import Counter

# Approximate: "Month Day Time hostname process[pid]: rest" -> process is the word before [ or :
proc_counts = Counter()
for line in raw_lines:
    line = line.rstrip()
    # Match hostname and process: ... hostname PROCESS[pid]: or hostname PROCESS:
    m = re.search(r"\S+\s+([^\[\s]+)(?:\[\d+\])?:\s", line)
    if m:
        proc_counts[m.group(1).strip()] += 1

print(f"Distinct process names: {len(proc_counts)}")
print()
for proc_name, count in proc_counts.most_common():
    print(f"  {proc_name:25s} {count:5d} ({count / len(raw_lines) * 100:5.1f}%)")

Distinct process names: 6

  CRON                        257 ( 94.5%)
  sshd                          4 (  1.5%)
  sudo                          4 (  1.5%)
  systemd-logind                3 (  1.1%)
  su                            3 (  1.1%)
  systemd                       1 (  0.4%)


### 2.2 Sample line per process type

Examine one example of each process type to understand message patterns.


In [9]:
# One example per process type
proc_examples = {}
for line in raw_lines:
    line = line.rstrip()
    m = re.search(r"\S+\s+([^\[\s]+)(?:\[\d+\])?:\s", line)
    if m:
        proc_name = m.group(1).strip()
        if proc_name not in proc_examples:
            proc_examples[proc_name] = line

for proc_name in ["CRON", "sshd", "sudo", "su", "systemd-logind", "systemd"]:
    if proc_name in proc_examples:
        print(f"--- {proc_name} ---")
        print(f"  {proc_examples[proc_name]}")
        print()

--- CRON ---
  Jan 23 06:25:05 intranet-server CRON[22883]: pam_unix(cron:session): session closed for user root

--- sshd ---
  Jan 23 16:23:04 intranet-server sshd[15014]: pam_unix(sshd:session): session closed for user jhall

--- sudo ---
  Jan 24 04:37:58 intranet-server sudo:    jhall : TTY=pts/1 ; PWD=/var/www/intranet.smith.russellmitchell.com/wp-content/uploads/2022/01 ; USER=root ; COMMAND=list

--- su ---
  Jan 24 04:37:40 intranet-server su[27950]: Successful su for jhall by www-data

--- systemd-logind ---
  Jan 23 16:23:04 intranet-server systemd-logind[957]: Removed session 111.

--- systemd ---
  Jan 23 16:30:47 intranet-server systemd: pam_unix(systemd-user:session): session opened for user jhall by (uid=0)



### 2.3 Parsing strategy

| Process | Message pattern | Parsed sub-fields (analysis only) |
|---------|-----------------|-----------------------------------|
| CRON | session opened/closed for user X by (uid=N) | message_user, message_uid |
| sshd | Accepted &lt;method&gt; for X from &lt;ip&gt; port &lt;port&gt; | message_user, message_remote_ip, message_remote_port, message_auth_method |
| sudo | TTY=... ; PWD=... ; USER=... ; COMMAND=... | message_tty, message_pwd, message_target_user, message_command |
| su | Successful su for X by Y | message_target_user, message_actor_user |
| systemd-logind | New/Removed session X of user Y | message_session_id |

The **raw** table stores only the full `message` as TEXT. Parsed `message_*` columns live in `df` for exploration; DDL matches `df_raw` (no `message_*`). Data quality: `.strip()` parsed sub-fields (trailing whitespace in TTY=, USER=, etc.).


In [10]:
import re
from datetime import UTC

import pandas as pd

# Syslog has no year; use dataset year (data_scope_and_findings.md: Jan 21-25, 2022)
TARGET_YEAR = 2022

# Match: Month Day Time hostname process[pid]: message
LINE_RX = re.compile(
    r"^([A-Z][a-z]{2})\s+(\d{1,2})\s+(\d{2}:\d{2}:\d{2})\s+(\S+)\s+([^\[\s]+)(?:\[(\d+)\])?:\s+(.*)$"
)


def parse_auth_line(line_num, line):
    # Parse one auth.log line. Returns dict with line_number, event_timestamp,
    # hostname, process_name, pid, message (raw blob), and message_* for analysis only.
    # All parsed message_* values are .strip()'d per findings data quality note.
    record = {"line_number": line_num}
    line = line.rstrip()
    m = LINE_RX.match(line)
    if not m:
        record["hostname"] = None
        record["process_name"] = None
        record["pid"] = None
        record["message"] = line.strip() if line else ""
        record["event_timestamp"] = None
        return record
    month, day, time_str, hostname, process_name, pid, message = m.groups()
    record["hostname"] = hostname
    record["process_name"] = process_name.strip() if process_name else None
    record["pid"] = int(pid) if pid else None
    record["message"] = message.strip() if message else ""

    # Timestamp
    ts_str = f"{TARGET_YEAR}-{month}-{int(day):02d} {time_str}"
    try:
        record["event_timestamp"] = pd.to_datetime(ts_str, format="%Y-%b-%d %H:%M:%S").tz_localize(
            UTC
        )
    except Exception:
        record["event_timestamp"] = None

    # --- Parsed sub-fields (message_*) for analysis only; do NOT put in raw DDL ---
    # session for user X / Accepted ... for X
    mu = re.search(r"(?:session (?:opened|closed) for user|Accepted \w+ for user) (\\S+)", message)
    if mu:
        record["message_user"] = mu.group(1).strip()
    # by (uid=N)
    muid = re.search(r"by \((?:uid=)?(\d+)\)", message)
    if muid:
        record["message_uid"] = int(muid.group(1))
    # sshd: from IP port N
    mip = re.search(r"from (\\d+\\.\\d+\\.\\d+\\.\\d+) port (\\d+)", message)
    if mip:
        record["message_remote_ip"] = mip.group(1).strip()
        record["message_remote_port"] = int(mip.group(2))
    # sshd: Accepted <method> for
    m_auth = re.search(r"Accepted (\\w+) for", message)
    if m_auth:
        record["message_auth_method"] = m_auth.group(1).strip()
    # sudo: TTY=... ; PWD=... ; USER=... ; COMMAND=...
    mty = re.search(r"TTY=([^;]+)", message)
    if mty:
        record["message_tty"] = mty.group(1).strip()
    mpwd = re.search(r"PWD=([^;]+)", message)
    if mpwd:
        record["message_pwd"] = mpwd.group(1).strip()
    muser = re.search(r"USER=([^;]+)", message)
    if muser:
        record["message_target_user"] = muser.group(1).strip()
    mcmd = re.search(r"COMMAND=(.+)$", message)
    if mcmd:
        record["message_command"] = mcmd.group(1).strip()
    # su: Successful su for X by Y
    msu = re.search(r"Successful su for (\\S+) by (\\S+)", message)
    if msu:
        record["message_target_user"] = msu.group(1).strip()
        record["message_actor_user"] = msu.group(2).strip()
    # systemd-logind: New session X of user Y / Removed session X
    mses = re.search(r"(?:New session|Removed session) ([^.]+)", message)
    if mses:
        record["message_session_id"] = mses.group(1).strip()

    return record


parsed = [parse_auth_line(i, line) for i, line in enumerate(raw_lines, 1)]
print(f"Parsed {len(parsed)} records")
print(f"Sample (line 1): {parsed[0]}")

Parsed 272 records
Sample (line 1): {'line_number': 1, 'hostname': 'intranet-server', 'process_name': 'CRON', 'pid': 22883, 'message': 'pam_unix(cron:session): session closed for user root', 'event_timestamp': Timestamp('2022-01-23 06:25:05+0000', tz='UTC')}


In [11]:
df = pd.DataFrame(parsed)

# pid as nullable int
if "pid" in df.columns:
    df["pid"] = df["pid"].astype("Int64")
if "message_remote_port" in df.columns:
    df["message_remote_port"] = df["message_remote_port"].astype("Int64")
if "message_uid" in df.columns:
    df["message_uid"] = df["message_uid"].astype("Int64")

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
msg_cols = [c for c in df.columns if c.startswith("message_")]
print(f"Parsed message_* columns (analysis only): {msg_cols}")

Shape: (272, 12)
Columns: ['line_number', 'hostname', 'process_name', 'pid', 'message', 'event_timestamp', 'message_uid', 'message_session_id', 'message_tty', 'message_pwd', 'message_target_user', 'message_command']
Parsed message_* columns (analysis only): ['message_uid', 'message_session_id', 'message_tty', 'message_pwd', 'message_target_user', 'message_command']


## 3. Field-by-Field Exploration

### 3.1 process_name (event family)


In [12]:
print("=== process_name distribution ===")
proc = df["process_name"].value_counts()
for name, count in proc.items():
    print(f"  {name:25s} {count:5d} ({count / len(df) * 100:5.1f}%)")

=== process_name distribution ===
  CRON                        257 ( 94.5%)
  sshd                          4 (  1.5%)
  sudo                          4 (  1.5%)
  systemd-logind                3 (  1.1%)
  su                            3 (  1.1%)
  systemd                       1 (  0.4%)


### 3.2 event_timestamp


In [13]:
print("=== event_timestamp range ===")
print(f"  Earliest: {df['event_timestamp'].min()}")
print(f"  Latest:   {df['event_timestamp'].max()}")
print()
print("=== Events per day ===")
dates = df["event_timestamp"].dt.date
for date, count in dates.value_counts().sort_index().items():
    print(f"  {date}  {count:5d}")

=== event_timestamp range ===
  Earliest: 2022-01-23 06:25:05+00:00
  Latest:   2022-01-24 23:39:01+00:00

=== Events per day ===
  2022-01-23    115
  2022-01-24    157


### 3.3 message_* coverage (parsed sub-fields)

These columns are **analysis-only**; they are not in the raw table DDL.


In [14]:
msg_cols = [c for c in df.columns if c.startswith("message_") and c != "message"]
if msg_cols:
    print("Coverage of parsed message_* columns:")
    for col in sorted(msg_cols):
        non_null = df[col].notna().sum()
        pct = non_null / len(df) * 100
        print(f"  {col:30s} {non_null:5d}/{len(df)} ({pct:5.1f}%)")
else:
    print("No message_* columns (besides message)")

Coverage of parsed message_* columns:
  message_command                    2/272 (  0.7%)
  message_pwd                        2/272 (  0.7%)
  message_session_id                 3/272 (  1.1%)
  message_target_user                2/272 (  0.7%)
  message_tty                        2/272 (  0.7%)
  message_uid                      132/272 ( 48.5%)


### 3.4 High-signal events (non-CRON)

Filter to interactive/security-relevant process names (sshd, sudo, su, systemd-logind, systemd).


In [15]:
non_cron = df[df["process_name"] != "CRON"]
print(f"Non-CRON lines: {len(non_cron)}")
print()
# Show key columns for high-signal events
cols = ["line_number", "event_timestamp", "process_name", "message"]
if "message_user" in df.columns:
    cols.append("message_user")
if "message_target_user" in df.columns:
    cols.append("message_target_user")
if "message_command" in df.columns:
    cols.append("message_command")
display_cols = [c for c in cols if c in non_cron.columns]
print(non_cron[display_cols].to_string())

Non-CRON lines: 15

     line_number           event_timestamp    process_name                                                                                                                                    message message_target_user       message_command
65            66 2022-01-23 16:23:04+00:00            sshd                                                                                      pam_unix(sshd:session): session closed for user jhall                 NaN                   NaN
66            67 2022-01-23 16:23:04+00:00  systemd-logind                                                                                                                       Removed session 111.                 NaN                   NaN
67            68 2022-01-23 16:30:46+00:00            sshd                   Accepted publickey for jhall from 172.19.131.174 port 49828 ssh2: RSA SHA256:8wFbiaYPevKS/wYKnePO20v0iymTcrRh4Kr+1uRS1UM                 NaN                   NaN
68            69 202

## 4. Raw 1:1 DataFrame

Build `df_raw` by dropping all `message_*` columns **except** `message`. The raw table DDL must match `df_raw` (truly raw: message as single TEXT blob).


In [16]:
message_star_cols = [c for c in df.columns if c.startswith("message_") and c != "message"]
df_raw = df.drop(columns=message_star_cols)

print(f"df shape: {df.shape}")
print(f"df_raw shape: {df_raw.shape}")
print(f"Raw columns: {list(df_raw.columns)}")
print()
print("Null counts in df_raw:")
for col in df_raw.columns:
    n = df_raw[col].isnull().sum()
    if n > 0:
        print(f"  {col}: {n}")

df shape: (272, 12)
df_raw shape: (272, 6)
Raw columns: ['line_number', 'hostname', 'process_name', 'pid', 'message', 'event_timestamp']

Null counts in df_raw:
  pid: 5


In [17]:
print("=== First 5 rows (df_raw) ===")
print(df_raw.head().to_string())
print()
print("=== Last 3 rows (df_raw) ===")
print(df_raw.tail(3).to_string())

=== First 5 rows (df_raw) ===
   line_number         hostname process_name    pid                                                          message           event_timestamp
0            1  intranet-server         CRON  22883             pam_unix(cron:session): session closed for user root 2022-01-23 06:25:05+00:00
1            2  intranet-server         CRON  23064  pam_unix(cron:session): session opened for user root by (uid=0) 2022-01-23 06:39:01+00:00
2            3  intranet-server         CRON  23064             pam_unix(cron:session): session closed for user root 2022-01-23 06:39:01+00:00
3            4  intranet-server         CRON  23137  pam_unix(cron:session): session opened for user root by (uid=0) 2022-01-23 06:47:01+00:00
4            5  intranet-server         CRON  23137             pam_unix(cron:session): session closed for user root 2022-01-23 06:47:02+00:00

=== Last 3 rows (df_raw) ===
     line_number         hostname process_name    pid                             

## 5. Ground Truth Labels

The label file maps line numbers to attack labels (JSONL). 8 labeled lines (145–152): privilege escalation (su to jhall by www-data, sudo commands including `cat /etc/shadow`).


In [18]:
import json

labels = []
if LABEL_FILE.exists():
    with open(LABEL_FILE) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            labels.append(json.loads(line))

print(f"Labeled records: {len(labels)}")
if labels:
    print(f"Labeled line numbers: {[lbl['line'] for lbl in labels]}")
    print()
    for lbl in labels:
        print(f"  line {lbl['line']}: labels={lbl['labels']}, rules={list(lbl['rules'].keys())}")
else:
    print("Label file not found or empty.")

Labeled records: 8
Labeled line numbers: [145, 146, 147, 148, 149, 150, 151, 152]

  line 145: labels=['attacker_change_user', 'escalate'], rules=['attacker_change_user', 'escalate']
  line 146: labels=['attacker_change_user', 'escalate', 'escalated_command', 'escalated_sudo_command'], rules=['attacker_change_user', 'escalate', 'escalated_command', 'escalated_sudo_command']
  line 147: labels=['attacker_change_user', 'escalate'], rules=['attacker_change_user', 'escalate']
  line 148: labels=['attacker_change_user', 'escalate'], rules=['attacker_change_user', 'escalate']
  line 149: labels=['escalated_command', 'escalated_sudo_command', 'escalate'], rules=['escalated_command', 'escalated_sudo_command', 'escalate']
  line 150: labels=['escalated_command', 'escalated_sudo_command', 'escalate', 'escalated_sudo_session'], rules=['escalated_command', 'escalated_sudo_command', 'escalate', 'escalated_sudo_session']
  line 151: labels=['escalated_command', 'escalated_sudo_command', 'escalated_s

In [19]:
# Cross-reference labeled lines with df_raw
if labels:
    labeled_line_nums = {lbl["line"] for lbl in labels}
    df_labeled = df_raw[df_raw["line_number"].isin(labeled_line_nums)]
    print(f"Labeled records matched in df_raw: {len(df_labeled)}")
    print()
    print(df_labeled[["line_number", "event_timestamp", "process_name", "message"]].to_string())
else:
    df_labeled = df_raw.iloc[0:0]
    print("No labels loaded.")

Labeled records matched in df_raw: 8

     line_number           event_timestamp    process_name                                                                                                                                    message
144          145 2022-01-24 04:37:40+00:00              su                                                                                                        Successful su for jhall by www-data
145          146 2022-01-24 04:37:40+00:00              su                                                                                                                + /dev/pts/1 www-data:jhall
146          147 2022-01-24 04:37:40+00:00              su                                                                            pam_unix(su:session): session opened for user jhall by (uid=33)
147          148 2022-01-24 04:37:40+00:00  systemd-logind                                                                                                              Ne

In [20]:
# Label and rule distribution (findings: escalate 8, attacker_change_user 4, etc.)
if labels:
    from collections import Counter

    label_counter = Counter()
    for lbl in labels:
        for name in lbl["labels"]:
            label_counter[name] += 1
    print("=== Label distribution ===")
    for name, count in label_counter.most_common():
        print(f"  {name}: {count}")
    print()
    print("=== Raw log lines for labeled events ===")
    for lbl in labels:
        line_num = lbl["line"]
        print(f"  [{line_num}] {raw_lines[line_num - 1].rstrip()}")
        print(f"         labels: {lbl['labels']}")
        print()

=== Label distribution ===
  escalate: 8
  escalated_command: 5
  escalated_sudo_command: 5
  attacker_change_user: 4
  escalated_sudo_session: 3

=== Raw log lines for labeled events ===
  [145] Jan 24 04:37:40 intranet-server su[27950]: Successful su for jhall by www-data
         labels: ['attacker_change_user', 'escalate']

  [146] Jan 24 04:37:40 intranet-server su[27950]: + /dev/pts/1 www-data:jhall
         labels: ['attacker_change_user', 'escalate', 'escalated_command', 'escalated_sudo_command']

  [147] Jan 24 04:37:40 intranet-server su[27950]: pam_unix(su:session): session opened for user jhall by (uid=33)
         labels: ['attacker_change_user', 'escalate']

  [148] Jan 24 04:37:40 intranet-server systemd-logind[957]: New session c1 of user jhall.
         labels: ['attacker_change_user', 'escalate']

  [149] Jan 24 04:37:58 intranet-server sudo:    jhall : TTY=pts/1 ; PWD=/var/www/intranet.smith.russellmitchell.com/wp-content/uploads/2022/01 ; USER=root ; COMMAND=list
  

## 6. Schema Mapping

Map the parsed fields to the planned **raw** table `auth_events_raw` (raw 1:1 loading, no normalization). The table stores `message` as a single TEXT blob (1NF violation). Parsed sub-fields (`message_*`) are for analysis only and are **not** in the raw DDL.

### 6.1 Raw field → column mapping

| # | Raw field | DB Column | PostgreSQL Type | MySQL Type | Nullable | Notes |
|---|-----------|-----------|-----------------|------------|----------|-------|
| 1 | *(auto)* | `auth_event_id` | `SERIAL PRIMARY KEY` | `INT AUTO_INCREMENT PRIMARY KEY` | No | Surrogate key |
| 2 | (line index) | `line_number` | `INTEGER NOT NULL` | `INT NOT NULL` | No | 1-based; candidate key; joins to labels |
| 3 | syslog timestamp | `event_timestamp` | `TIMESTAMP WITH TIME ZONE` | `DATETIME` | Yes | Parsed from line (no year in syslog; use dataset year) |
| 4 | hostname | `hostname` | `VARCHAR(100)` | `VARCHAR(100)` | Yes | Always `intranet-server` in this file |
| 5 | process name | `process_name` | `VARCHAR(50) NOT NULL` | `VARCHAR(50) NOT NULL` | No | CRON, sshd, sudo, su, systemd-logind, systemd |
| 6 | PID | `pid` | `INTEGER` | `INT` | Yes | From `[pid]` when present (missing on some sudo/systemd lines) |
| 7 | full message | `message` | `TEXT NOT NULL` | `TEXT NOT NULL` | No | Raw blob (1NF violation: embeds TTY, PWD, USER, COMMAND, IP, etc.) |

Optional columns when loading with labels (see `hunt_auth_logs_findings.md`): `auth_event_category` (TEXT[] / JSON), `auth_signature_matches` (JSONB / JSON) store labels and rules as-is; normalization unpacks them into `attack_labels`.

### 6.2 Raw DDL


In [21]:
# PostgreSQL: core raw table (matches df_raw)
postgresql_ddl = """
-- PostgreSQL
CREATE TABLE auth_events_raw (
    auth_event_id   SERIAL PRIMARY KEY,
    line_number     INTEGER NOT NULL,
    event_timestamp TIMESTAMP WITH TIME ZONE,
    hostname        VARCHAR(100),
    process_name    VARCHAR(50) NOT NULL,
    pid             INTEGER,
    message         TEXT NOT NULL,
    auth_event_category    TEXT[],
    auth_signature_matches JSONB,
    created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

# MySQL
mysql_ddl = """
-- MySQL
CREATE TABLE auth_events_raw (
    auth_event_id   INT AUTO_INCREMENT PRIMARY KEY,
    line_number     INT NOT NULL,
    event_timestamp DATETIME,
    hostname        VARCHAR(100),
    process_name    VARCHAR(50) NOT NULL,
    pid             INT,
    message         TEXT NOT NULL,
    auth_event_category    JSON,
    auth_signature_matches JSON,
    created_at      DATETIME DEFAULT CURRENT_TIMESTAMP
);
"""

print(postgresql_ddl)
print(mysql_ddl)


-- PostgreSQL
CREATE TABLE auth_events_raw (
    auth_event_id   SERIAL PRIMARY KEY,
    line_number     INTEGER NOT NULL,
    event_timestamp TIMESTAMP WITH TIME ZONE,
    hostname        VARCHAR(100),
    process_name    VARCHAR(50) NOT NULL,
    pid             INTEGER,
    message         TEXT NOT NULL,
    auth_event_category    TEXT[],
    auth_signature_matches JSONB,
    created_at      TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);


-- MySQL
CREATE TABLE auth_events_raw (
    auth_event_id   INT AUTO_INCREMENT PRIMARY KEY,
    line_number     INT NOT NULL,
    event_timestamp DATETIME,
    hostname        VARCHAR(100),
    process_name    VARCHAR(50) NOT NULL,
    pid             INT,
    message         TEXT NOT NULL,
    auth_event_category    JSON,
    auth_signature_matches JSON,
    created_at      DATETIME DEFAULT CURRENT_TIMESTAMP
);



## 7. Observations for Normalization Phase

Use the `normalization_rules_sheet.md` checklist.

### 7.1 1NF Check

- **Multi-valued / composite field:** The `message` column often contains multiple attributes in one cell (e.g. sudo: `TTY=... ; PWD=... ; USER=... ; COMMAND=...`; sshd: remote IP, port, auth method). **1NF violated.** Resolution: unpack into separate columns during normalization.
- **Repeating groups:** None in this table (no phone1/phone2-style columns).
- **Checklist:** Multi-valued field identified (message); resolution direction: parse into atomic columns in normalized schema.

### 7.2 2NF Check

- The table has a **single-column primary key** (`auth_event_id`). Partial dependencies require a composite key, so **2NF is satisfied** (single-column PK → no partial dependencies possible).

### 7.3 3NF Check

- **Transitive dependency:** `process_name` determines which subset of attributes can be parsed from `message` (CRON vs sshd vs sudo vs su vs systemd-logind). So `auth_event_id` → `process_name` → populated field set. **3NF violated.** Resolution: subtype tables or type-specific columns in the final 3NF schema (e.g. parent `log_events` + subtype `auth_events`).

### 7.4 Preliminary Functional Dependencies

| FD | Determinant | Dependent(s) | Reasoning |
|----|-------------|---------------|-----------|
| FD1 | `auth_event_id` | all other attributes | Surrogate PK. |
| FD2 | `line_number` | all other attributes | Each log line is unique; candidate key. |
| FD3 | `process_name` | populated field set | Process family determines which message sub-fields exist (3NF violation). |
| FD4 | `hostname` | constant in this file | Single host; meaningful after unioning multiple sources. |


## 8. Summary

### What we discovered

1. `auth.log` has 272 lines; syslog format with `hostname process[pid]: message`.
2. **1NF violated:** `message` is a composite blob (TTY, PWD, USER, COMMAND, IP, etc. embedded).
3. **2NF satisfied:** single-column PK.
4. **3NF violated:** `process_name` determines which fields are meaningful (type → field set).
5. CRON dominates (~94.5%); high-signal events are sshd, sudo, su (15 non-CRON lines). 8 labeled lines (145–152) are privilege escalation (su to jhall by www-data, sudo including `cat /etc/shadow`).

### Raw schema output

- `df_raw`: 7 columns (line_number, event_timestamp, hostname, process_name, pid, message) — DDL matches this; table name `auth_events_raw`. Optional label columns `auth_event_category`, `auth_signature_matches` when loading with labels.
- Parsed `message_*` columns live in `df` for analysis only; they are not in the raw table.

### Next steps

- Consolidate findings in `docs/data_exploration/notebook_findings/hunt_auth_logs_findings.md`.
- In normalization: unpack `message` into atomic columns; resolve type → field_set via subtype or derived columns; optionally merge with audit.log for a unified auth_events table.
